In [1]:
import math
import scipy.optimize as spo
import numpy as np
import os
import matplotlib.pyplot as plt

def psi_cosineLE(N_points=100):
    ang = np.linspace(0,np.pi/2,N_points)
    psi = -np.cos(ang)+1
    return psi

def Kulfan_eval(coeff, psi, zeta_T):
    n = coeff.size - 1
    N1 = 0.5
    N2 = 1.0
    C = psi**(N1)*(1-psi)**(N2)
    S = np.zeros([n+1, psi.size])
    S_temp = np.zeros([n+1, psi.size])
    zeta_temp = np.zeros([n+1, psi.size])
    for i in range(0,n+1):
        S[i,:]=math.factorial(n)/(math.factorial(i)*math.factorial(n-i))  *  psi**i  *  (1-psi)**(n-i)
        S_temp[i,:] = coeff[i]*S[i,:]
        zeta_temp[i,:] = C*S_temp[i,:]+psi*zeta_T
    Sf = np.sum(S_temp, axis=0)
    zeta_coeff = psi**(N1)*(1-psi)**(N2)*Sf+psi*zeta_T
    return zeta_coeff

def Kulfan_residual(coeff, *args):
    psi,zeta,zeta_T = args
    n = coeff.size - 1
    N1 = 0.5
    N2 = 1.0
    C = psi**(N1)*(1-psi)**(N2)
    S = np.zeros([n+1, psi.size])
    S_temp = np.zeros([n+1, psi.size])
    zeta_temp = np.zeros([n+1, psi.size])
    for i in range(0,n+1):
        S[i,:]=math.factorial(n)/(math.factorial(i)*math.factorial(n-i))  *  psi**i  *  (1-psi)**(n-i)
        S_temp[i,:] = coeff[i]*S[i,:]
        zeta_temp[i,:] = C*S_temp[i,:]+psi*zeta_T
    Sf = np.sum(S_temp, axis=0)
    zeta_coeff = psi**(N1)*(1-psi)**(N2)*Sf+psi*zeta_T
    return zeta-zeta_coeff

fls = os.listdir('.')
fls = [f for f in fls if f.endswith('.dat') and 'CST' not in f]

for fl in fls:
    plt.figure(figsize=(22,4))
    data = np.genfromtxt(fl)

    flip = False
    upper_data = []
    lower_data = []
    for dr in data:
        if flip:
            lower_data.append(dr)
        else:
            upper_data.append(dr)
        if dr[1] == 0 and flip == False:
            flip = True

    upper_data = np.array(upper_data)
    lower_data = np.array(lower_data)

    coeff_guess = np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])
    resu=spo.leastsq(Kulfan_residual, coeff_guess.tolist(), args=(upper_data[:,0], upper_data[:,1], upper_data[0,1]) )
    resl=spo.leastsq(Kulfan_residual, (-1*coeff_guess).tolist(), args=(lower_data[:,0], lower_data[:,1], lower_data[-1,1]) )

    psi = psi_cosineLE(100)
    zeta_T_lower = lower_data[-1,1]
    zeta_T_upper = upper_data[0,1]

    K_upper = Kulfan_eval(resu[0], psi, zeta_T_upper)
    K_lower = Kulfan_eval(resl[0], psi, zeta_T_lower)

    xvals = np.append(list(reversed(psi)), psi[1:])
    yvals = np.append(list(reversed(K_upper)), K_lower[1:])

    np.savetxt('../fitted/'+fl.replace('.dat','_fittedCST10.dat'), np.array([xvals, yvals]).T)

    plt.title(fl)
    plt.plot(upper_data[:,0], upper_data[:,1], 'x', label=fl+' upper')
    plt.plot(lower_data[:,0], lower_data[:,1], 'x', label=fl+' lower')
    plt.plot(psi, K_upper, '.-', label=fl+' upper CST10')
    plt.plot(psi, K_lower, '.-', label=fl+' lower CST10')
    plt.grid(1)
    plt.legend()
    plt.tight_layout()
    plt.axis('equal')
    plt.savefig('../fitted/plot_'+fl.replace('.dat','_CST10fit.png'), dpi=300)
    plt.close()